# DaFitMatch embedder on Colab (free GPU)

Runs the **offline** batch embedder: products, reference looks, archetype centroids and venue vibes.
No model ever runs at request time (ADR 0002) - this notebook only fills the vectors.

1. Runtime -> Change runtime type -> **T4 GPU**
2. Fill in the secrets below (never paste keys in a cell):
   - `SUPABASE_URL`
   - `SUPABASE_SECRET_KEY` (service role)
3. Run the cells top to bottom.

In [ ]:
!git clone https://github.com/<your-org>/DaFitMatch.git
%cd DaFitMatch/workers/embedder
!pip install -q -r requirements.txt
print("installed")

In [ ]:
import os

from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_SECRET_KEY"] = userdata.get("SUPABASE_SECRET_KEY")
os.environ.setdefault("EMBEDDER_DEVICE", "cuda")
print("secrets loaded")

In [ ]:
# 1) reference looks (run first: the centroids need them)
!python -m embedder embed-reference-looks --manifest looks/manifest.json
!python -m embedder compute-centroids

In [ ]:
# 2) the catalog (resumable: re-run to pick up where it stopped)
!python -m embedder embed-products --batch-size 128

In [ ]:
# 3) venue vibes (text encoder)
!python -m embedder embed-venues

In [ ]:
# 4) users' own wardrobe (private bucket, signed URLs, photos never leave it)
!python -m embedder embed-wardrobe

## Notes

- The job is **resumable**: rows with an embedding are skipped, so re-running continues the batch.
- `--all` re-embeds everything (only after changing the model - that needs a new ADR).
- Keep `looks/images/` out of git; the manifest lists where the images are.
- If writes fail with connection resets, your ISP may be blocking `*.supabase.co` (see ADR 0008).